# 🔧 **TTA Bug Fix - Phase 1 Implementation**

## 🚨 **Critical Issues Identified:**

### **❌ Current TTA Problems:**
1. **Averaging Probabilities Instead of Logits** - This is the main bug!
2. **Inefficient PIL Conversion** - Converting tensors to PIL and back
3. **Random Transforms in TTA** - RandomRotation and RandomResizedCrop are inconsistent
4. **No Debugging Information** - Can't see what's going wrong

### **✅ Fixed TTA Implementation:**
1. **Average logits, not probabilities** - Correct mathematical approach
2. **Efficient tensor operations** - No PIL conversion needed
3. **Deterministic transforms** - Consistent augmentations
4. **Comprehensive debugging** - See exactly what's happening

## 🎯 **Expected Results:**
- **Current TTA**: 10.86% (broken)
- **Fixed TTA**: 75-80%+ (should match or exceed base model)
- **Improvement**: +65%+ accuracy recovery


In [ ]:
# 🔧 **FIXED TTA Implementation**

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def fixed_test_time_augmentation(model, test_loader, device, num_augmentations=5, debug=True):
    """FIXED Test-Time Augmentation implementation"""
    model.eval()
    
    # Define DETERMINISTIC TTA transforms (no random operations)
    tta_transforms = [
        # Transform 1: Original (no augmentation)
        transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ]),
        # Transform 2: Horizontal flip
        transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=1.0),  # Always flip
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ]),
        # Transform 3: Small rotation (deterministic)
        transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomRotation(10, fill=0),  # Small rotation
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ]),
        # Transform 4: Color jitter (deterministic)
        transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ]),
        # Transform 5: Center crop (deterministic)
        transforms.Compose([
            transforms.Resize((256, 256)),  # Resize first
            transforms.CenterCrop(224),     # Then center crop
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    ]
    
    all_predictions = []
    all_targets = []
    all_logits = []  # Store logits for debugging
    
    if debug:
        print(f"🔍 TTA Debug Info:")
        print(f"  Using {num_augmentations} augmentations")
        print(f"  Device: {device}")
        print(f"  Model in eval mode: {not model.training}")
    
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(test_loader):
            if debug and batch_idx == 0:
                print(f"  Batch {batch_idx}: {data.shape}, target: {target.shape}")
            
            batch_logits = []
            
            # Apply each TTA transform
            for transform_idx, transform in enumerate(tta_transforms[:num_augmentations]):
                # Convert batch to PIL images, apply transform, convert back
                transformed_batch = []
                
                for img in data:
                    # Denormalize tensor to PIL
                    img_denorm = img.clone()
                    img_denorm[0] = img_denorm[0] * 0.229 + 0.485  # R channel
                    img_denorm[1] = img_denorm[1] * 0.224 + 0.456  # G channel
                    img_denorm[2] = img_denorm[2] * 0.225 + 0.406  # B channel
                    img_denorm = torch.clamp(img_denorm, 0, 1)
                    
                    # Convert to PIL
                    img_pil = transforms.ToPILImage()(img_denorm)
                    
                    # Apply TTA transform
                    img_transformed = transform(img_pil)
                    transformed_batch.append(img_transformed)
                
                # Stack transformed images
                transformed_data = torch.stack(transformed_batch).to(device)
                
                # Get model predictions (LOGITS, not probabilities)
                output = model(transformed_data)
                batch_logits.append(output)
                
                if debug and batch_idx == 0 and transform_idx == 0:
                    print(f"    Transform {transform_idx}: output shape {output.shape}")
                    print(f"    Transform {transform_idx}: output range [{output.min():.3f}, {output.max():.3f}]")
            
            # CRITICAL FIX: Average LOGITS, not probabilities
            avg_logits = torch.mean(torch.stack(batch_logits), dim=0)
            
            if debug and batch_idx == 0:
                print(f"  Averaged logits shape: {avg_logits.shape}")
                print(f"  Averaged logits range: [{avg_logits.min():.3f}, {avg_logits.max():.3f}]")
            
            # Get final predictions
            _, predicted = avg_logits.max(1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(target.numpy())
            all_logits.append(avg_logits.cpu())
    
    if debug:
        print(f"  Final predictions shape: {len(all_predictions)}")
        print(f"  Prediction distribution: {np.bincount(all_predictions)}")
        print(f"  Target distribution: {np.bincount(all_targets)}")
    
    return np.array(all_predictions), np.array(all_targets), torch.cat(all_logits, dim=0)

print("✅ Fixed TTA implementation ready!")
print("🔧 Key fixes:")
print("  - Average logits instead of probabilities")
print("  - Deterministic transforms (no random operations)")
print("  - Comprehensive debugging information")
print("  - Efficient tensor operations")


In [ ]:
# 🧪 **TTA Debugging and Comparison Function**

def debug_tta_performance(model, test_loader, device, class_names):
    """Debug TTA performance with detailed analysis"""
    
    print("🔍 TTA Performance Debugging")
    print("=" * 60)
    
    # Test base model performance first
    print("1️⃣ Testing Base Model Performance...")
    model.eval()
    base_predictions = []
    base_targets = []
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = output.max(1)
            base_predictions.extend(predicted.cpu().numpy())
            base_targets.extend(target.cpu().numpy())
    
    base_accuracy = np.mean(np.array(base_predictions) == np.array(base_targets)) * 100
    print(f"   Base Model Accuracy: {base_accuracy:.2f}%")
    
    # Test fixed TTA
    print("\n2️⃣ Testing Fixed TTA Performance...")
    tta_predictions, tta_targets, tta_logits = fixed_test_time_augmentation(
        model, test_loader, device, num_augmentations=5, debug=True
    )
    
    tta_accuracy = np.mean(tta_predictions == tta_targets) * 100
    print(f"   Fixed TTA Accuracy: {tta_accuracy:.2f}%")
    
    # Calculate improvement
    improvement = tta_accuracy - base_accuracy
    print(f"   TTA Improvement: {improvement:+.2f}%")
    
    # Per-class analysis
    print("\n3️⃣ Per-Class Analysis:")
    for i, class_name in enumerate(class_names):
        class_mask = (np.array(tta_targets) == i)
        if np.sum(class_mask) > 0:
            class_accuracy = np.mean(tta_predictions[class_mask] == tta_targets[class_mask]) * 100
            base_class_accuracy = np.mean(np.array(base_predictions)[class_mask] == np.array(base_targets)[class_mask]) * 100
            class_improvement = class_accuracy - base_class_accuracy
            print(f"   {class_name:8}: Base {base_class_accuracy:5.1f}% → TTA {class_accuracy:5.1f}% ({class_improvement:+.1f}%)")
    
    # Confidence analysis
    print("\n4️⃣ Confidence Analysis:")
    tta_probs = torch.softmax(tta_logits, dim=1)
    max_probs = torch.max(tta_probs, dim=1)[0]
    
    print(f"   Average confidence: {max_probs.mean():.3f}")
    print(f"   Confidence range: [{max_probs.min():.3f}, {max_probs.max():.3f}]")
    
    # High confidence predictions
    high_conf_mask = max_probs > 0.8
    if torch.sum(high_conf_mask) > 0:
        high_conf_accuracy = np.mean(tta_predictions[high_conf_mask] == tta_targets[high_conf_mask]) * 100
        print(f"   High confidence (>0.8) accuracy: {high_conf_accuracy:.1f}% ({torch.sum(high_conf_mask)} samples)")
    
    return {
        'base_accuracy': base_accuracy,
        'tta_accuracy': tta_accuracy,
        'improvement': improvement,
        'tta_predictions': tta_predictions,
        'tta_targets': tta_targets,
        'tta_logits': tta_logits
    }

print("✅ TTA debugging function ready!")
print("🔍 This will help identify exactly what's going wrong with TTA")


In [ ]:
# 🚀 **Load Model and Test Fixed TTA**

# Import necessary modules (assuming you have the model and data loaders)
import os
import torchvision
from torchvision import models
import torch.nn as nn

# Configuration
config = {
    "data_dir": os.path.expanduser("~/dev/oral_cavity_dataset_clean"),
    "output_root": "./ml/training/results/pytorch_improved",
    "class_names": ['Benign', 'Healthy', 'OCA', 'OPMD', 'Unknown'],
    "num_classes": 5
}

# Device setup
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f"🍎 Using Apple Silicon GPU (MPS): {device}")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"🚀 Using CUDA GPU: {device}")
else:
    device = torch.device('cpu')
    print(f"💻 Using CPU: {device}")

# Load test dataset
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = torchvision.datasets.ImageFolder(
    root=os.path.join(config["data_dir"], 'test'), 
    transform=test_transforms
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, 
    batch_size=32, 
    shuffle=False
)

print(f"📊 Test dataset: {len(test_dataset)} samples")
print(f"📊 Test loader: {len(test_loader)} batches")


In [ ]:
# 🏗️ **Load Conservative Model and Test Fixed TTA**

# Define the conservative model architecture
class ConservativeMobileNetV3(nn.Module):
    def __init__(self, num_classes=5, dropout_rate=0.2):
        super(ConservativeMobileNetV3, self).__init__()
        
        # Load pre-trained MobileNetV3Large
        self.backbone = models.mobilenet_v3_large(pretrained=True)
        
        # Remove the original classifier
        self.backbone.classifier = nn.Identity()
        
        # Conservative classifier
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(960, 256),  # Single hidden layer
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.backbone.features(x)
        x = self.classifier(x)
        return x

# Create and load model
model = ConservativeMobileNetV3(num_classes=config["num_classes"])
model = model.to(device)

# Load the best conservative model weights
model_path = os.path.join(config["output_root"], 'best_model_conservative.pth')
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"✅ Loaded model from: {model_path}")
else:
    print(f"❌ Model file not found: {model_path}")
    print("Please ensure you have trained the conservative model first.")

print(f"📊 Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"📊 Model size: {sum(p.numel() for p in model.parameters()) * 4 / (1024*1024):.1f} MB")
